# z604 - Features Mi Serie (Etapa 4)
Ratios leave-one-out contra grupos (CAT1, CAT2, CAT3, marca, mi_serie=descripcion) + tamano anterior/posterior.
Fuente: `tb_features_FE601.parquet` (Etapa 2) + `tb_productos.txt`.

In [ ]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [ ]:
PARAM = {
    'experimento': 'FE604',
    'features_path': './exp/FE601/tb_features_FE601.parquet',
    'productos_path': './buckets/b1/datasets/tb_productos.txt',
    'grupos': [['cat1'], ['cat2'], ['cat3'], ['brand']],
    'metricas': ['tn', 'tn_media_12']
}

ruta = os.path.join('./exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

## 1. Cargar y unir con tb_productos

In [ ]:
df = pl.read_parquet(PARAM['features_path'])

tb_productos = pl.read_csv(PARAM['productos_path'], separator="\t").select(
    ["product_id", "cat1", "cat2", "cat3", "brand", "sku_size", "descripcion"]
)

df = df.join(tb_productos, on="product_id", how="left")
print(df.height, df.width)

## 2. Ratios leave-one-out contra grupos
Para cada (grupo, periodo): promedio del grupo EXCLUYENDO al propio producto. Ratio = mi_valor / promedio_resto_grupo.

Funcion generica -- agregar un grupo o metrica nueva es una linea en `PARAM`, no codigo nuevo.

In [ ]:
def ratio_leave_one_out(df, group_cols, metric):
    nombre_grupo = "_".join(group_cols)
    agg = df.group_by(group_cols + ["periodo"]).agg(
        pl.col(metric).sum().alias("_suma_grupo"),
        pl.len().alias("_n_grupo")
    )
    out = df.join(agg, on=group_cols + ["periodo"], how="left")
    out = out.with_columns(
        (
            (pl.col("_suma_grupo") - pl.col(metric))
            / (pl.col("_n_grupo") - 1).clip(lower_bound=1)
        ).alias(f"{metric}_prom_{nombre_grupo}_excl")
    )
    out = out.with_columns(
        (pl.col(metric) / (pl.col(f"{metric}_prom_{nombre_grupo}_excl") + 1e-6)).alias(
            f"ratio_{metric}_{nombre_grupo}"
        )
    )
    return out.drop(["_suma_grupo", "_n_grupo", f"{metric}_prom_{nombre_grupo}_excl"])

for grupo in PARAM['grupos']:
    for metrica in PARAM['metricas']:
        df = ratio_leave_one_out(df, grupo, metrica)

## 3. Mi Serie (mismo producto, distintas presentaciones)
Agrupado por `descripcion`. Incluye el ratio leave-one-out (igual que los grupos anteriores) y el vecino de tamano (anterior/posterior en `sku_size`).

In [ ]:
for metrica in PARAM['metricas']:
    df = ratio_leave_one_out(df, ["descripcion"], metrica)

In [ ]:
# vecinos de tamano dentro de mi_serie (por descripcion, ordenado por sku_size)
vecinos = tb_productos.sort(["descripcion", "sku_size"]).with_columns([
    pl.col("product_id").shift(1).over("descripcion").alias("product_id_tam_ant"),
    pl.col("product_id").shift(-1).over("descripcion").alias("product_id_tam_post"),
]).select(["product_id", "product_id_tam_ant", "product_id_tam_post"])

df = df.join(vecinos, on="product_id", how="left")

In [ ]:
# self-join: traigo el tn del vecino de tamano, mismo periodo
tn_por_producto_periodo = df.select(["product_id", "periodo", "tn"])

df = df.join(
    tn_por_producto_periodo.rename({"product_id": "product_id_tam_ant", "tn": "tn_tam_ant"}),
    on=["product_id_tam_ant", "periodo"],
    how="left"
)
df = df.join(
    tn_por_producto_periodo.rename({"product_id": "product_id_tam_post", "tn": "tn_tam_post"}),
    on=["product_id_tam_post", "periodo"],
    how="left"
)

df = df.drop(["product_id_tam_ant", "product_id_tam_post"])

## 4. Guardar

In [ ]:
salida = os.path.join(ruta, "tb_features_FE604.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)